# Ensemble JPTs

This tutorial introduces the three ensemble learners shipped in `jpt.ensembles`:

* **`JPTForest`** — *bagging*: a uniform mixture of JPTs trained on bootstrap
  resamples. Fully generative.
* **`JPTLikelihoodBoost`** — *generative boosting*: greedily grows a convex
  mixture $P_t = (1-\alpha_t)\,P_{t-1} + \alpha_t\,C_t$ where every new
  component up-weights the samples the current mixture explains worst.
  Fully generative.
* **`JPTBoost`** — *discriminative gradient boosting* (regression or softmax
  classification). Models $E[y \mid x]$ resp. $P(y \mid x)$ only — it gives
  up the joint.

The first two are subclasses of `MixtureJPT`. Because a convex mixture of
normalized joints is again a normalized joint, they support the **complete JPT
query surface** — `likelihood`, `infer`, `posterior`, `expectation`, `mpe`,
`sample`, and `conditional_jpt` — with formulas that reduce to per-member
single-tree queries.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

from jpt.ensembles import JPTForest, JPTLikelihoodBoost, JPTBoost

iris = load_iris(as_frame=True)
df = iris.data.copy()
df.columns = [c.replace(' (cm)', '').replace(' ', '_') for c in df.columns]
df['species'] = iris.target.map(dict(enumerate(iris.target_names)))

rng = np.random.RandomState(42)
test_mask = rng.rand(len(df)) < .25
train, test = df[~test_mask], df[test_mask]
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Bagging: `JPTForest`

A forest is trained like a single `JPT`: variables are inferred from the
DataFrame automatically. Each member fits the *integer bootstrap
multiplicities* of the data natively via `sample_weight` — no resampled
DataFrames are materialized.

In [2]:
forest = JPTForest(
    n_estimators=10,
    min_samples_leaf=.1,
    random_state=0
)
forest.learn(train)
print(len(forest), 'members,', sum(len(m.leaves) for m in forest.members), 'leaves total')

10 members, 83 leaves total


### The generative query surface

All single-tree queries carry over. Unconditional functionals distribute
linearly over the members; conditional queries use the evidence-adjusted
*responsibilities* $r_m(e) \propto w_m P^{(m)}(e)$.

In [3]:
# Per-row joint density and held-out mean log-likelihood
print('mean log L =', forest.log_likelihood(test) / len(test))

# P(species = setosa | sepal_length < 5.5)
print('P =', forest.infer(
    query={'species': 'setosa'},
    evidence={'sepal_length': [0, 5.5]}
))

# Posterior distribution of petal_length given the species
posterior = forest.posterior(
    variables=['petal_length'],
    evidence={'species': 'virginica'}
)
print('E[petal_length | virginica] =', posterior['petal_length'].expectation())

# Most probable explanation given partial evidence
mpe, likelihood = forest.mpe(evidence={'species': 'setosa'})
print('MPE:', mpe)

# Ancestral sampling from the mixture
forest.sample(5)

mean log L = -0.20347623185268476
P = 0.8149260942456914
E[petal_length | virginica] = 5.3987569511402675
MPE: [<LabelAssignment {sepal_length: <ContinuousSet=[5.100,5.200)>, sepal_width: <ContinuousSet=[3.400,3.500)>, petal_length: <ContinuousSet=[1.400,1.500)>, petal_width: <ContinuousSet=[0.200,0.300)>, species: {np.str_('setosa')}}>]


array([[np.float64(6.7267894430173625), np.float64(3.1530278079824225),
        np.float64(5.042638290638291), np.float64(1.561316745434298),
        np.str_('virginica')],
       [np.float64(5.577421256817038), np.float64(2.5399175013109576),
        np.float64(3.956709653290097), np.float64(1.177012895023726),
        np.str_('versicolor')],
       [np.float64(6.7696374261694885), np.float64(2.9297599611776493),
        np.float64(6.76133040455122), np.float64(2.2760549562320245),
        np.str_('virginica')],
       [np.float64(5.360809896037443), np.float64(2.545409517460316),
        np.float64(3.6840065465334555), np.float64(1.2185788915044717),
        np.str_('versicolor')],
       [np.float64(5.899287747760444), np.float64(2.445038215224069),
        np.float64(4.828525121498826), np.float64(2.1874661036188248),
        np.str_('virginica')]], dtype=object)

### Closure under conditioning

Conditioning a mixture on evidence $e$ yields *again* a mixture: each member
is conditioned with the single-tree routine and the weights become the
responsibilities. The result is a full `MixtureJPT` — chainable and
supporting every query above.

In [4]:
conditional = forest.conditional_jpt(evidence={'species': 'versicolor'})
print(type(conditional).__name__, 'with', len(conditional), 'members')
print('E[petal_width | versicolor] =',
      conditional.expectation(['petal_width'])['petal_width'])

MixtureJPT with 10 members
E[petal_width | versicolor] = 1.315104199979119


## Generative boosting: `JPTLikelihoodBoost`

Instead of independent members, components are added greedily: round $t$
trains a new JPT on a resample that up-weights the points the current mixture
assigns low density, then takes the optimal convex step towards it (exact
concave line search). The mixture remains a valid normalized joint at every
round — no partition function is ever computed.

In [5]:
booster = JPTLikelihoodBoost(
    n_rounds=6,
    min_samples_leaf=.2,
    random_state=0
)
booster.learn(train)
print('components:', len(booster), 'weights:', np.round(booster.weights, 3))
print('held-out mean log L =', booster.log_likelihood(test) / len(test))

components: 7 weights: [0.158 0.172 0.116 0.159 0.112 0.139 0.144]


held-out mean log L = -0.1292521686044634


`JPTLikelihoodBoost` is a `MixtureJPT` like the forest, so the entire query
surface of the previous section — including `conditional_jpt` — applies
unchanged.

## Discriminative boosting: `JPTBoost`

`JPTBoost` is *not* a mixture: it is a classic additive gradient-boosting
expansion $F_T(x) = F_0 + \nu \sum_t h_t(x)$ with weak JPTs as base
learners. The mode is selected from the target dtype: numeric targets get
squared-error regression, non-numeric ones softmax classification.

**The trade-off:** an additive expansion tilts the model *multiplicatively*
in probability space, which would require an intractable partition function
over the whole data space — unless one conditions on $x$ first, which
collapses it to a per-row normalization. `JPTBoost` therefore models only the
conditional and offers `predict` / `predict_proba` instead of the generative
queries. Use the mixtures above whenever you need the joint.

In [6]:
clf = JPTBoost(
    target='species',
    n_rounds=40,
    learning_rate=.3,
    min_samples_leaf=.1
)
clf.learn(train)
accuracy = (clf.predict(test) == test['species'].to_numpy()).mean()
print('accuracy =', round(accuracy, 3))
clf.predict_proba(test.head(3)).round(3)

accuracy = 0.952


array([[0.939, 0.03 , 0.03 ],
       [0.939, 0.03 , 0.03 ],
       [0.939, 0.03 , 0.03 ]])

In [7]:
# The same class does regression when the target column is numeric:
reg = JPTBoost(
    target='petal_length',
    n_rounds=40,
    learning_rate=.3,
    min_samples_leaf=.1
)
reg.learn(train.drop(columns='species'))
prediction = reg.predict(test.drop(columns='species'))
residual = prediction - test['petal_length'].to_numpy()
print('RMSE =', round(float(np.sqrt(np.mean(residual ** 2))), 3))

RMSE = 0.37


## Serialization

All ensembles follow the package-wide `to_json` / `from_json` protocol, and
pickling routes through it — persist with either.

In [8]:
import pickle

clone = JPTForest.from_json(forest.to_json())
assert clone == forest

clone = pickle.loads(pickle.dumps(booster))
assert clone == booster

clone = JPTBoost.from_json(clf.to_json())
assert (clone.predict(test) == clf.predict(test)).all()
print('round trips OK')

round trips OK


## Which ensemble should I use?

* You need **joint-distribution queries** (`infer`, `posterior`, `mpe`,
  `sample`, conditioning) with lower variance than a single tree →
  **`JPTForest`**.
* You care about **density estimation quality** (held-out likelihood) →
  **`JPTLikelihoodBoost`**.
* You only need **point predictions or class probabilities** for one fixed
  target and want maximal predictive accuracy → **`JPTBoost`**.